In [ ]:
import sys
print(sys.executable)

import pandas, sklearn
print(pandas.__version__, sklearn.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
df.shape

df.columns.to_list()


In [ ]:
df['Class'].value_counts(normalize=True)

#df.info()

In [ ]:
df.duplicated().sum()


In [ ]:
df.describe()

In [ ]:
print(df['Time'].head())

print(df['Time'].min()) 
print(df['Time'].max())


In [ ]:
df['hour_of_day'] = (df['Time'] % 86400) // 3600
print(df[['Time', 'hour_of_day']].head())

In [ ]:
print(df['hour_of_day'].unique())

df.groupby('hour_of_day')['Class'].mean()

In [ ]:
df.groupby('hour_of_day')['Class'].sum()

In [ ]:
#all dupicate rows 
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10) 

In [ ]:
df = df.drop_duplicates()
df.shape

In [ ]:
df.groupby('Class')['Amount'].describe()

In [ ]:
sns.boxplot(x='Class', y='Amount', data=df[df['Amount'] < 500])
plt.show()

In [ ]:
# only catches linear relationships
correlations = df.corr()['Class'].sort_values(ascending=False)
correlations

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

In [ ]:
X_train.filter(like='V').skew()

In [ ]:
from sklearn.preprocessing import RobustScaler

cols_to_scale = X_train.columns.to_list()
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled.describe()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

print(classification_report(y_test, y_pred_dummy))

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
logreg.fit(X_train_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_scaled)

print(classification_report(y_test, y_pred_logreg))

In [ ]:
# show acutal probs from sklearn before it applied 0.5 threshold
y_proba = logreg.predict_proba(X_test_scaled)
y_proba[:5] #logreg.classes to check field order


In [ ]:
#fraud only
y_proba_fraud = y_proba[:,1]
y_proba_fraud[:5]

In [ ]:
threshold = 0.3
y_pred_custom = (y_proba_fraud >= threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_pred_t = (y_proba_fraud >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_proba_fraud) # this does automatically, at a much finer resolution, exactly what 9 threshold maunal loop did

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Logistic Regression')
plt.show()

In [ ]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_proba_fraud)
print(f"PR-AUC: {pr_auc: .3f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [ ]:
pr_auc_rf = average_precision_score(y_test, y_proba_rf)
print(f"Random Forest PR-AUC: {pr_auc_rf: .3f}")

In [ ]:
# classification report needs 0 or 1, not probas
y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

In [ ]:
# back to using proba vs 0/1 to fine tune
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_pred_t = (y_proba_rf >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

In [ ]:
precisions_rf, recalls_rf, _ = precision_recall_curve(y_test, y_proba_rf)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, label=f'Logistic Regression (PR-AUC={pr_auc:.3f})')
plt.plot(recalls_rf, precisions_rf, label=f'Random Forest (PR-AUC={pr_auc_rf:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve Comparison')
plt.legend()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index= X_train.columns)

importances_sort = importances.sort_values(ascending=False)
importances_sort